# SC-16-Homomorphic-Encryption - Chiffrement Homomorphique

**Navigation** : [Index](../README.md) | [<< Zero-Knowledge Proofs](SC-15-Zero-Knowledge-Proofs.ipynb) | [E2E Voting >>](SC-17-E2E-Verifiable-Voting.ipynb)

---

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Comprendre le **chiffrement homomorphique** (PHE, SHE, FHE) et ses variantes
2. Implementer le schema de **Paillier** (additively homomorphic) avec `phe`
3. Explorer le schema **CKKS** pour l'arithmetique approchee avec TenSEAL
4. Decouvrir le **calcul multipartite securise** (MPC) et le partage de secrets de Shamir

### Prerequis

- Python 3.10+
- `phe` (python-paillier) : `pip install phe`
- `tenseal` (optionnel) : `pip install tenseal`
- `mpyc` (optionnel) : `pip install mpyc`

### Duree estimee : 50 minutes

---

## 1. Concepts du chiffrement homomorphique

Le **chiffrement homomorphique** (HE) permet d'effectuer des calculs directement
sur des donnees chiffrees, sans les dechiffrer. Le resultat chiffre, une fois dechiffre,
correspond exactement au resultat qu'on aurait obtenu en calculant sur les donnees en clair.

### Formalisation

Soit `E` la fonction de chiffrement et `D` la fonction de dechiffrement :

$$D(E(a) \oplus E(b)) = a + b$$

ou $\oplus$ designe l'operation homomorphique dans l'espace chiffre.

### Les trois niveaux de chiffrement homomorphique

| Type | Sigle | Operations | Profondeur | Exemples |
|------|-------|-----------|-----------|----------|
| Partially Homomorphic | **PHE** | Addition **ou** multiplication, pas les deux | Illimitee (1 operation) | RSA (mult.), Paillier (add.), ElGamal (mult.) |
| Somewhat Homomorphic | **SHE** | Addition **et** multiplication | Limitee (bruit croissant) | BGV, BFV |
| Fully Homomorphic | **FHE** | Toutes operations, profondeur arbitraire | Illimitee (bootstrapping) | TFHE, CKKS (approche), Concrete |

### Applications concretes

- **Vote electronique** : le serveur additionne les votes chiffres sans voir les bulletins individuels
- **ML confidentiel** : entrainer ou inferer un modele sur des donnees chiffrees (sante, finance)
- **Cloud computing** : deleguer un calcul a un serveur non-fiable sans reveler les donnees
- **Genomique** : analyser des sequences ADN chiffrees pour le diagnostic medical

### Historique

| Annee | Avancee | Auteur |
|-------|---------|--------|
| 1978 | RSA : multiplication homomorphique | Rivest, Shamir, Adleman |
| 1999 | Paillier : addition homomorphique | Pascal Paillier |
| 2009 | Premier schema FHE (bootstrapping) | Craig Gentry |
| 2016 | CKKS : FHE pour nombres reels | Cheon, Kim, Kim, Song |
| 2020+ | Concrete (Zama) : FHE compile depuis Python | Zama.ai |

In [ ]:
# Vue d'ensemble des schemas de chiffrement homomorphique
schemas = {
    "RSA (1978)": {
        "type": "PHE",
        "operation": "Multiplication",
        "principe": "E(a) * E(b) = E(a * b) mod n",
        "usage": "Signatures, echange de cles",
    },
    "Paillier (1999)": {
        "type": "PHE",
        "operation": "Addition",
        "principe": "E(a) * E(b) = E(a + b) mod n^2",
        "usage": "Vote, statistiques agregatrices",
    },
    "BGV/BFV (2011-12)": {
        "type": "SHE/FHE",
        "operation": "Addition + Multiplication (entiers)",
        "principe": "Lattice-based, bruit gere par modular switching",
        "usage": "Calculs exacts sur entiers",
    },
    "CKKS (2016)": {
        "type": "FHE (approche)",
        "operation": "Addition + Multiplication (reels)",
        "principe": "Encodage dans l'erreur, calcul approche",
        "usage": "ML confidentiel, statistiques",
    },
    "TFHE (2016)": {
        "type": "FHE",
        "operation": "Bootstrapping rapide (portes logiques)",
        "principe": "LWE + bootstrapping par porte",
        "usage": "Circuits arbitraires, Concrete (Zama)",
    },
}

print("SCHEMAS DE CHIFFREMENT HOMOMORPHIQUE")
print("=" * 70)
for name, info in schemas.items():
    print(f"\n{name} [{info['type']}]")
    print(f"  Operation : {info['operation']}")
    print(f"  Principe  : {info['principe']}")
    print(f"  Usage     : {info['usage']}")

### Observation

Le point cle est que chaque schema fait un **compromis** entre :
- La **richesse des operations** supportees (PHE < SHE < FHE)
- La **performance** (PHE est rapide, FHE est 10 000x plus lent que le calcul en clair)
- La **precision** (CKKS est approche, BGV/BFV sont exacts)

Dans la suite, nous implementons Paillier (PHE) car c'est le plus simple et le plus
pertinent pour le vote electronique (section 6).

---

## 2. Chiffrement de Paillier (PHE additif)

Le schema de **Paillier** (1999) est un chiffrement a cle publique avec la propriete :

$$E(m_1) \cdot E(m_2) \mod n^2 = E(m_1 + m_2)$$

Autrement dit, **multiplier deux chiffres** dans l'espace chiffre revient a **additionner
les messages en clair**. On peut aussi multiplier un chiffre par un scalaire en clair :

$$E(m)^k \mod n^2 = E(k \cdot m)$$

### Construction simplifiee

1. **Generation** : choisir deux grands premiers $p, q$. Poser $n = p \cdot q$
2. **Chiffrement** : $c = g^m \cdot r^n \mod n^2$ (r aleatoire)
3. **Dechiffrement** : utiliser $\lambda = \text{lcm}(p-1, q-1)$ pour retrouver $m$

La bibliotheque `phe` (python-paillier) implemente tout cela.

In [ ]:
# Installation si necessaire
# !pip install phe

from phe import paillier
import time

# Generation des cles
print("GENERATION DES CLES PAILLIER")
print("=" * 70)
start = time.time()
public_key, private_key = paillier.generate_paillier_keypair(n_length=2048)
elapsed = time.time() - start
print(f"Taille de n : {public_key.n.bit_length()} bits")
print(f"Temps de generation : {elapsed:.3f}s")
print(f"n (tronque) : {str(public_key.n)[:40]}...")

In [ ]:
# Chiffrement et dechiffrement de base
print("CHIFFREMENT / DECHIFFREMENT")
print("=" * 70)

# Chiffrer des entiers
valeurs = [42, 100, -7, 0, 999999]
for v in valeurs:
    chiffre = public_key.encrypt(v)
    dechiffre = private_key.decrypt(chiffre)
    # Le chiffre est un grand nombre, on en montre juste un extrait
    c_str = str(chiffre.ciphertext())[:30]
    print(f"  {v:>8} -> E({v}) = {c_str}... -> D = {dechiffre}")

print()
print("Propriete importante : chaque chiffrement est ALEATOIRE (semantiquement sur)")
a1 = public_key.encrypt(42)
a2 = public_key.encrypt(42)
print(f"  E(42) premiere fois  : {str(a1.ciphertext())[:30]}...")
print(f"  E(42) deuxieme fois  : {str(a2.ciphertext())[:30]}...")
print(f"  Chiffres identiques ? {a1.ciphertext() == a2.ciphertext()} (toujours False)")

In [ ]:
# Propriete homomorphique : addition sur les chiffres
print("ADDITION HOMOMORPHIQUE")
print("=" * 70)

a = 42
b = 58
c = 100

# Chiffrer
ea = public_key.encrypt(a)
eb = public_key.encrypt(b)
ec = public_key.encrypt(c)

# Addition dans l'espace chiffre
e_sum_ab = ea + eb            # E(42) + E(58) = E(100)
e_sum_abc = ea + eb + ec      # E(42) + E(58) + E(100) = E(200)

# Dechiffrer les resultats
sum_ab = private_key.decrypt(e_sum_ab)
sum_abc = private_key.decrypt(e_sum_abc)

print(f"  a = {a}, b = {b}, c = {c}")
print(f"  D(E(a) + E(b))     = {sum_ab}  (attendu: {a + b})")
print(f"  D(E(a) + E(b) + E(c)) = {sum_abc}  (attendu: {a + b + c})")
print()

# Multiplication par un scalaire en clair
print("MULTIPLICATION PAR SCALAIRE")
print("-" * 40)
k = 7
e_prod = ea * k              # E(42) * 7 = E(294)
prod = private_key.decrypt(e_prod)
print(f"  D(E({a}) * {k})      = {prod}  (attendu: {a * k})")
print()

# Combinaison : somme ponderee
weights = [3, 5, 2]
values = [10, 20, 30]
encrypted_values = [public_key.encrypt(v) for v in values]
encrypted_weighted = sum(ev * w for ev, w in zip(encrypted_values, weights))
result = private_key.decrypt(encrypted_weighted)
expected = sum(v * w for v, w in zip(values, weights))
print(f"SOMME PONDEREE")
print(f"  Valeurs  : {values}")
print(f"  Poids    : {weights}")
print(f"  D(sum(E(vi)*wi)) = {result}  (attendu: {expected})")

### Interpretation : proprietes de Paillier

| Operation | Formule | Support |
|-----------|---------|--------|
| Addition chiffre + chiffre | `E(a) + E(b) = E(a+b)` | Oui |
| Multiplication chiffre x scalaire | `E(a) * k = E(a*k)` | Oui |
| Somme ponderee | `sum(E(vi) * wi)` | Oui |
| Multiplication chiffre x chiffre | `E(a) * E(b) = E(a*b)` | **Non** |
| Comparaison | `E(a) > E(b) ?` | **Non** |

Paillier est **additively homomorphic** : il ne supporte que l'addition
(et la multiplication par scalaire, qui est une addition repetee).
C'est suffisant pour le vote, les moyennes, les sommes statistiques.

---

## 3. CKKS avec TenSEAL (FHE pour nombres reels)

Le schema **CKKS** (Cheon-Kim-Kim-Song, 2016) permet des operations sur des
**nombres reels** (virgule flottante) avec une approximation controlee.
C'est le schema prefere pour le **machine learning confidentiel**.

**TenSEAL** est une bibliotheque Python qui expose CKKS (et BFV) avec une API
compatible tenseurs.

### Principe de CKKS

1. Les nombres reels sont **encodes** dans des polynomes cyclotomiques
2. Le bruit de chiffrement est **inclus dans la precision** (pas un defaut, une feature)
3. Chaque operation multiplie le bruit ; le **bootstrapping** le reinitialise
4. La precision est configurable (~40 bits pour des calculs ML classiques)

In [ ]:
# TenSEAL : CKKS pour l'arithmetique approchee
# pip install tenseal

try:
    import tenseal as ts
    import numpy as np

    # Creer un contexte CKKS
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,
        coeff_mod_bit_sizes=[60, 40, 40, 60]
    )
    context.generate_galois_keys()
    context.global_scale = 2**40

    print("CKKS AVEC TENSEAL")
    print("=" * 70)
    print(f"Degree du polynome : {8192}")
    print(f"Scale (precision)  : 2^40")
    print()

    # Chiffrer des vecteurs
    v1 = [1.5, 2.3, 4.7, 8.1]
    v2 = [0.5, 1.7, 3.3, 1.9]

    enc_v1 = ts.ckks_vector(context, v1)
    enc_v2 = ts.ckks_vector(context, v2)

    print(f"v1 (clair)  : {v1}")
    print(f"v2 (clair)  : {v2}")
    print()

    # Addition chiffree
    enc_add = enc_v1 + enc_v2
    result_add = enc_add.decrypt()
    expected_add = [a + b for a, b in zip(v1, v2)]
    print("Addition chiffree :")
    print(f"  Resultat   : {[round(x, 6) for x in result_add]}")
    print(f"  Attendu    : {expected_add}")
    print(f"  Erreur max : {max(abs(r - e) for r, e in zip(result_add, expected_add)):.2e}")
    print()

    # Multiplication chiffree
    enc_mul = enc_v1 * enc_v2
    result_mul = enc_mul.decrypt()
    expected_mul = [a * b for a, b in zip(v1, v2)]
    print("Multiplication chiffree :")
    print(f"  Resultat   : {[round(x, 6) for x in result_mul]}")
    print(f"  Attendu    : {expected_mul}")
    print(f"  Erreur max : {max(abs(r - e) for r, e in zip(result_mul, expected_mul)):.2e}")
    print()

    # Produit scalaire chiffre (operation ML fondamentale)
    enc_dot = enc_v1.dot(enc_v2)
    result_dot = enc_dot.decrypt()
    expected_dot = sum(a * b for a, b in zip(v1, v2))
    print("Produit scalaire chiffre :")
    print(f"  Resultat   : {round(result_dot[0], 6)}")
    print(f"  Attendu    : {expected_dot}")
    print(f"  Erreur     : {abs(result_dot[0] - expected_dot):.2e}")

except ImportError:
    print("TenSEAL non installe.")
    print("Pour installer : pip install tenseal")
    print()
    print("TenSEAL permet de manipuler des vecteurs chiffres avec le schema CKKS.")
    print("Les operations supportees incluent addition, multiplication, et produit scalaire.")
    print("L'erreur d'approximation est typiquement de l'ordre de 1e-6 a 1e-4.")

In [ ]:
# Performance et profondeur multiplicative CKKS

try:
    import tenseal as ts
    import time

    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,
        coeff_mod_bit_sizes=[60, 40, 40, 60]
    )
    context.global_scale = 2**40

    # Mesurer les temps d'operation
    v = [3.14] * 100
    enc_v = ts.ckks_vector(context, v)

    print("BENCHMARK CKKS (vecteur de 100 elements)")
    print("=" * 70)

    # Temps de chiffrement
    start = time.time()
    for _ in range(100):
        _ = ts.ckks_vector(context, v)
    t_encrypt = (time.time() - start) / 100
    print(f"  Chiffrement        : {t_encrypt*1000:.2f} ms")

    # Temps d'addition
    enc_v2 = ts.ckks_vector(context, v)
    start = time.time()
    for _ in range(100):
        _ = enc_v + enc_v2
    t_add = (time.time() - start) / 100
    print(f"  Addition chiffree  : {t_add*1000:.2f} ms")

    # Temps de multiplication
    start = time.time()
    for _ in range(100):
        _ = enc_v * enc_v2
    t_mul = (time.time() - start) / 100
    print(f"  Multiplication     : {t_mul*1000:.2f} ms")

    # Temps de dechiffrement
    start = time.time()
    for _ in range(100):
        _ = enc_v.decrypt()
    t_decrypt = (time.time() - start) / 100
    print(f"  Dechiffrement      : {t_decrypt*1000:.2f} ms")

    print()
    print("  Overhead vs calcul en clair : ~1000x-10000x")
    print("  -> Le HE est couteux, a utiliser quand la confidentialite l'exige")

except ImportError:
    print("TenSEAL non installe - benchmark ignore.")
    print()
    print("Ordres de grandeur typiques CKKS (vecteur 100 elements) :")
    print("  Chiffrement       : ~1-5 ms")
    print("  Addition chiffree : ~0.1 ms")
    print("  Multiplication    : ~1-5 ms")
    print("  Overhead total    : ~1000x-10000x vs calcul en clair")

### Interpretation : CKKS vs Paillier

| Critere | Paillier (PHE) | CKKS (FHE approche) |
|---------|---------------|--------------------|
| Operations | Addition seulement | Addition + Multiplication |
| Types de donnees | Entiers | Nombres reels (approches) |
| Precision | Exacte | ~40 bits (configurable) |
| Performance | Rapide (PHE) | Plus lent (FHE) |
| Profondeur | Illimitee (addition) | Limitee par coeff_mod |
| Usage principal | Vote, sommes | ML, statistiques |

CKKS est le choix natural pour le **machine learning confidentiel** car il supporte
les multiplications necessaires aux reseaux de neurones (produits matriciels).

---

## 4. FHE avec Concrete (Zama)

**Concrete** de Zama est un framework qui permet de **compiler du code Python en FHE** :
on ecrit une fonction Python normale, et Concrete la transforme en un circuit FHE
executable sur des donnees chiffrees.

### Principe

1. Ecrire une fonction Python avec des operations sur entiers
2. Concrete **trace** la fonction et genere un circuit FHE
3. Le circuit est compile et optimise pour le bootstrapping TFHE
4. Execution : les donnees sont chiffrees, le circuit calcule, le resultat est dechiffre

### Limites actuelles

- Operations sur **entiers** seulement (pas de flottants)
- Taille des entiers limitee (8, 16, 32 bits)
- Temps de compilation long pour des circuits complexes
- Performance ~10 000x plus lente que le calcul en clair

In [ ]:
# Concrete : compiler Python en FHE
# pip install concrete-python

try:
    from concrete import fhe
    import numpy as np

    # Definir une fonction simple
    def addition_secrete(x, y):
        return x + y

    # Compiler en circuit FHE
    compiler = fhe.Compiler(
        addition_secrete,
        {"x": "encrypted", "y": "encrypted"}
    )

    # Donner des exemples pour la compilation
    inputset = [(np.uint8(i), np.uint8(j)) for i in range(10) for j in range(10)]
    circuit = compiler.compile(inputset)

    print("CONCRETE FHE - Addition compilee")
    print("=" * 70)
    print(f"Circuit compile avec succes")

    # Generer les cles
    circuit.keygen()

    # Executer sur des donnees chiffrees
    a, b = 7, 13
    result = circuit.encrypt_run_decrypt(a, b)
    print(f"  addition_secrete({a}, {b}) = {result}  (attendu: {a + b})")
    print()
    print("-> Le serveur a calcule {a} + {b} SANS voir les valeurs !")

except ImportError:
    print("Concrete non installe (pip install concrete-python).")
    print()
    print("Concrete permet de compiler une fonction Python en circuit FHE.")
    print("Exemple conceptuel :")
    print()
    print("  @fhe.compiler({'x': 'encrypted', 'y': 'encrypted'})")
    print("  def addition_secrete(x, y):")
    print("      return x + y")
    print()
    print("  # Compilation")
    print("  circuit = addition_secrete.compile(inputset)")
    print("  circuit.keygen()")
    print()
    print("  # Execution sur donnees chiffrees")
    print("  result = circuit.encrypt_run_decrypt(7, 13)  # -> 20")
    print("  # Le serveur n'a jamais vu 7 ni 13")

except Exception as e:
    print(f"Erreur lors de la compilation Concrete : {e}")
    print("Concrete necessite un environnement specifique (Linux recommande).")

### Observation : etat de l'art FHE

Le FHE est un domaine en **evolution rapide**. Les performances doublent
environ tous les 18 mois (analogue a la loi de Moore pour le HE).

| Framework | Schema | Langage | Specialite |
|-----------|--------|---------|------------|
| **Concrete** (Zama) | TFHE | Python | Compilation Python -> FHE |
| **TenSEAL** | CKKS, BFV | Python | ML confidentiel |
| **OpenFHE** | BGV, BFV, CKKS, TFHE | C++ | Reference academique |
| **SEAL** (Microsoft) | BFV, CKKS | C++ | Cloud computing |
| **HElib** (IBM) | BGV, CKKS | C++ | Historique |

Le defi principal reste la **performance** : un circuit FHE typique est
10 000 a 100 000 fois plus lent que le calcul en clair.

---

## 5. Calcul multipartite securise (MPC) et partage de secrets

Le **MPC** (Multi-Party Computation) est une alternative au HE pour le calcul
confidentiel. Au lieu de chiffrer les donnees et calculer sur le chiffre,
on **distribue les donnees** entre plusieurs parties qui collaborent pour
calculer un resultat sans reveler leurs entrees individuelles.

### Partage de secrets de Shamir (1979)

**Adi Shamir** a invente un schema de **partage de secrets** a seuil :
un secret $s$ est divise en $n$ parts, et il faut au moins $k$ parts
pour le reconstituer (schema $(k, n)$).

**Principe mathematique** : un polynome de degre $k-1$ est defini par $k$ points.

1. Choisir un polynome aleatoire $P(x) = s + a_1 x + a_2 x^2 + ... + a_{k-1} x^{k-1}$
   ou $P(0) = s$ est le secret
2. Distribuer les parts : part $i$ = $P(i)$ pour $i = 1, ..., n$
3. Reconstruction : interpolation de Lagrange avec $k$ parts quelconques

In [ ]:
import random
from functools import reduce

# Arithmetique modulaire pour eviter les problemes de precision
# On travaille dans Z/pZ avec p premier
PRIME = 2**127 - 1  # 12e nombre premier de Mersenne


def shamir_split(secret, k, n, prime=PRIME):
    """Partager un secret en n parts avec un seuil de k.

    Args:
        secret: le secret (entier)
        k: nombre minimum de parts pour reconstruire
        n: nombre total de parts
        prime: module premier pour l'arithmetique

    Returns:
        Liste de n paires (x, y) representant les parts
    """
    if k > n:
        raise ValueError("k doit etre <= n")

    # Generer un polynome aleatoire de degre k-1 avec P(0) = secret
    coefficients = [secret % prime] + [
        random.randrange(1, prime) for _ in range(k - 1)
    ]

    # Evaluer le polynome en x = 1, 2, ..., n
    def evaluate(x):
        result = 0
        power = 1
        for coeff in coefficients:
            result = (result + coeff * power) % prime
            power = (power * x) % prime
        return result

    shares = [(i, evaluate(i)) for i in range(1, n + 1)]
    return shares


def shamir_reconstruct(shares, prime=PRIME):
    """Reconstruire le secret a partir de k parts (interpolation de Lagrange).

    Args:
        shares: liste de paires (x, y)
        prime: module premier

    Returns:
        Le secret reconstruit
    """
    k = len(shares)

    def mod_inverse(a, p):
        """Inverse modulaire via le petit theoreme de Fermat."""
        return pow(a, p - 2, p)

    secret = 0
    for j in range(k):
        xj, yj = shares[j]
        # Calcul du coefficient de Lagrange L_j(0)
        numerator = 1
        denominator = 1
        for m in range(k):
            if m != j:
                xm = shares[m][0]
                numerator = (numerator * (-xm)) % prime
                denominator = (denominator * (xj - xm)) % prime

        lagrange = (numerator * mod_inverse(denominator, prime)) % prime
        secret = (secret + yj * lagrange) % prime

    return secret


# Demonstration
print("PARTAGE DE SECRETS DE SHAMIR")
print("=" * 70)

secret = 42
k = 3  # seuil
n = 5  # nombre de parts

print(f"Secret : {secret}")
print(f"Schema : ({k}, {n}) - il faut {k} parts sur {n} pour reconstruire")
print()

shares = shamir_split(secret, k, n)
print("Parts generees :")
for i, (x, y) in enumerate(shares):
    print(f"  Part {i+1} (x={x}) : {str(y)[:30]}...")
print()

In [ ]:
# Reconstruction avec differents sous-ensembles de parts
import itertools

print("RECONSTRUCTION AVEC DIFFERENTS SOUS-ENSEMBLES")
print("=" * 70)

# Avec exactement k=3 parts
print(f"\nAvec k={k} parts (minimum requis) :")
for combo in itertools.combinations(range(n), k):
    subset = [shares[i] for i in combo]
    recovered = shamir_reconstruct(subset)
    parts_used = [f"Part {i+1}" for i in combo]
    status = "OK" if recovered == secret else "ECHEC"
    print(f"  {parts_used} -> {recovered} [{status}]")

# Avec k-1 parts (insuffisant)
print(f"\nAvec k-1={k-1} parts (insuffisant) :")
for combo in itertools.combinations(range(n), k - 1):
    subset = [shares[i] for i in combo]
    recovered = shamir_reconstruct(subset)
    parts_used = [f"Part {i+1}" for i in combo]
    status = "OK" if recovered == secret else "INCORRECT"
    print(f"  {parts_used} -> {recovered} [{status}]")
    if combo == list(itertools.combinations(range(n), k - 1))[2]:
        print(f"  ... (toutes les combinaisons de {k-1} parts donnent un mauvais resultat)")
        break

print()
print(f"-> Avec {k} parts sur {n}, le secret est TOUJOURS reconstruit correctement")
print(f"-> Avec moins de {k} parts, le secret est INDEDUCTIBLE")

### Interpretation : HE vs MPC

| Critere | Chiffrement Homomorphique | MPC (Secret Sharing) |
|---------|--------------------------|---------------------|
| Modele de confiance | Un serveur non-fiable | Plusieurs parties semi-honnetes |
| Communication | Faible (une fois) | Elevee (echanges multiples) |
| Performance | Lent (calcul lourd) | Rapide si peu de parties |
| Flexibilite | Toutes operations (FHE) | Lineaire natif, non-lineaire couteux |
| Setup | Cles HE (lourd) | Distribution de parts (leger) |

**Quand utiliser quoi ?**
- **HE** : un seul serveur effectue le calcul (cloud, vote centralise)
- **MPC** : plusieurs parties collaborent (encheres, statistiques inter-entreprises)
- **Hybride** : combiner les deux pour le meilleur des deux mondes

---

## 6. Exercice : Systeme de sondage anonyme avec Paillier

Implementez un systeme de sondage anonyme ou :
1. Chaque participant chiffre sa reponse (note de 1 a 10) avec la cle publique Paillier
2. Le serveur calcule la **somme homomorphique** des votes chiffres
3. Seul le **resultat agrege** est dechiffre (moyenne, somme)
4. Aucun vote individuel n'est revele

### Squelette a completer

In [ ]:
from phe import paillier


class SondageAnonyme:
    """Systeme de sondage anonyme utilisant le chiffrement de Paillier.

    Le serveur peut calculer la somme et la moyenne des reponses
    sans jamais voir les reponses individuelles.
    """

    def __init__(self):
        """Generer les cles Paillier.
        La cle publique est distribuee aux participants.
        La cle privee est detenue par l'autorite de depouillement.
        """
        # TODO: Generer la paire de cles Paillier
        # self.public_key, self.private_key = ...
        raise NotImplementedError("Generez les cles Paillier")

    def chiffrer_vote(self, note):
        """Un participant chiffre sa note (1-10) avec la cle publique.

        Args:
            note: entier entre 1 et 10

        Returns:
            Le vote chiffre (EncryptedNumber)
        """
        # TODO:
        # 1. Valider que la note est entre 1 et 10
        # 2. Chiffrer avec self.public_key
        # 3. Retourner le vote chiffre
        raise NotImplementedError("Chiffrez le vote avec Paillier")

    def aggreger_votes(self, votes_chiffres):
        """Le serveur calcule la somme homomorphique (sans dechiffrer).

        Args:
            votes_chiffres: liste de votes chiffres (EncryptedNumber)

        Returns:
            La somme chiffree et le nombre de votes
        """
        # TODO:
        # 1. Calculer la somme des votes chiffres (utiliser l'addition homomorphique)
        # 2. Retourner (somme_chiffree, nombre_de_votes)
        raise NotImplementedError("Calculez la somme homomorphique")

    def depouiller(self, somme_chiffree, nombre_votes):
        """L'autorite dechiffre uniquement le resultat agrege.

        Args:
            somme_chiffree: la somme homomorphique chiffree
            nombre_votes: le nombre total de votes

        Returns:
            Dictionnaire avec somme, moyenne, nombre de votes
        """
        # TODO:
        # 1. Dechiffrer la somme avec self.private_key
        # 2. Calculer la moyenne
        # 3. Retourner {"somme": ..., "moyenne": ..., "nb_votes": ...}
        raise NotImplementedError("Dechiffrez le resultat agrege")


# === Test (decommentez apres implementation) ===
# sondage = SondageAnonyme()
#
# # Simulation : 10 participants votent
# import random
# notes = [random.randint(1, 10) for _ in range(10)]
# print(f"Notes en clair (pour verification) : {notes}")
# print(f"Somme attendue : {sum(notes)}, Moyenne attendue : {sum(notes)/len(notes):.2f}")
# print()
#
# # Chaque participant chiffre son vote
# votes_chiffres = [sondage.chiffrer_vote(note) for note in notes]
# print(f"Votes chiffres : {len(votes_chiffres)} votes (le serveur ne voit que des chiffres)")
#
# # Le serveur agregge sans dechiffrer
# somme_chiffree, nb = sondage.aggreger_votes(votes_chiffres)
# print(f"Somme chiffree calculee par le serveur (sans voir les votes)")
#
# # L'autorite dechiffre le resultat
# resultat = sondage.depouiller(somme_chiffree, nb)
# print(f"\nResultat du sondage :")
# print(f"  Somme   : {resultat['somme']}")
# print(f"  Moyenne : {resultat['moyenne']:.2f}")
# print(f"  Votes   : {resultat['nb_votes']}")

---

## 7. Resume

### Tableau comparatif des approches de calcul confidentiel

| Approche | Operations | Performance | Modele de confiance | Cas d'usage principal |
|----------|-----------|-------------|--------------------|-----------------------|
| **PHE (Paillier)** | Addition | Rapide | 1 serveur | Vote, sommes agregatrices |
| **SHE (BGV/BFV)** | Add + Mult (limitee) | Moyen | 1 serveur | Calculs entiers bornes |
| **FHE (CKKS)** | Toutes (approchees) | Lent | 1 serveur | ML confidentiel |
| **FHE (TFHE)** | Toutes (exactes) | Tres lent | 1 serveur | Circuits arbitraires |
| **MPC (Shamir)** | Lineaire natif | Variable | N parties | Statistiques partagees |

### Points cles

1. Le chiffrement homomorphique permet de **calculer sans dechiffrer** : c'est un changement de paradigme pour la confidentialite
2. **Paillier** (PHE) est simple et rapide mais limite a l'addition : ideal pour le vote et les aggregations
3. **CKKS** (FHE) permet le ML confidentiel mais avec un cout en performance de 10 000x
4. Le **partage de secrets de Shamir** distribue la confiance : aucun participant seul ne connait le secret
5. HE et MPC sont **complementaires**, pas concurrents : le choix depend du modele de confiance

---

**Notebook suivant** : [SC-17-E2E-Verifiable-Voting](SC-17-E2E-Verifiable-Voting.ipynb) - Combiner zero-knowledge proofs et chiffrement homomorphique pour un systeme de vote electronique verifiable de bout en bout